In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1NuF3_JCy2Y9MPPtK9O4yUCRLpZ9dSjKY", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


# Column-Parallel and Row-Parallel Linear Layers from Scratch

*Part 1 of the Vizuara Tensor Parallelism series*  
*Estimated time: 40-50 minutes*

In this notebook, you will implement the two fundamental building blocks of tensor parallelism:

1. **Column-parallel linear layers** -- split the weight matrix along columns, each GPU computes a slice of the output
2. **Row-parallel linear layers** -- split the weight matrix along rows, each GPU computes a partial sum, then allreduce

We will simulate multiple GPUs using plain Python/PyTorch on a single device. By the end, you will understand *exactly* how tensor parallelism decomposes matrix multiplications and why the math works out.

**Prerequisites:** Basic PyTorch, matrix multiplication, understanding of linear layers in neural networks.

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** -- it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/gpu-programming/tensor-parallelism/practice)**

In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import torch
import torch.nn as nn
import numpy as np

# We'll simulate tensor parallelism on a single device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Set random seed for reproducibility
torch.manual_seed(42)
print("\nSetup complete!")

---


In [ ]:
#@title 🎧 Transition: Section1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_section1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 1: Review -- What Does a Linear Layer Do?

A linear layer (without bias) computes:

$$Y = X W$$

where:
- $X$ has shape $(b, k)$ -- batch of $b$ tokens, each with $k$ features
- $W$ has shape $(k, n)$ -- the weight matrix
- $Y$ has shape $(b, n)$ -- the output

In a transformer MLP, the first linear layer projects from hidden dimension $h$ to intermediate dimension $4h$, so $k = h$ and $n = 4h$.

Let us start by establishing our baseline with a concrete example.

In [ ]:
#@title 🎧 Code Walkthrough: Linear Example Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_linear_example_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Our running example: a small linear layer
b = 2   # batch size (number of tokens)
k = 8   # input features
n = 12  # output features

# Create input and weight matrix
torch.manual_seed(42)
X = torch.randn(b, k, device=device)
W = torch.randn(k, n, device=device)

# Standard (non-parallel) forward pass
Y_full = X @ W

print(f"Input X shape:  {X.shape}")
print(f"Weight W shape: {W.shape}")
print(f"Output Y shape: {Y_full.shape}")
print(f"\nY_full =\n{Y_full}")

In [ ]:
#@title 🎧 Listen: Linear Example Output
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_linear_example_output.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


This is our reference answer. Every parallel strategy we implement must produce *exactly* this same output `Y_full`.


In [ ]:
#@title 🎧 Transition: Section2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_section2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
## Section 2: Column-Linear Parallelism

### The Idea

We split the weight matrix $W$ along its **column dimension** into $T$ pieces:

$$W = \begin{bmatrix} W_1 & W_2 & \cdots & W_T \end{bmatrix}$$

Each $W_i$ has shape $(k, n/T)$. GPU $i$ stores only $W_i$.

The computation decomposes as:

$$Y = X W = X \begin{bmatrix} W_1 & W_2 & \cdots & W_T \end{bmatrix} = \begin{bmatrix} X W_1 & X W_2 & \cdots & X W_T \end{bmatrix}$$

Each GPU computes $Y_i = X W_i$ independently, producing shape $(b, n/T)$. The full output is obtained by concatenating all partial outputs.

**Key properties:**
- Input $X$ must be replicated on every GPU
- No communication during the forward pass
- Outputs are naturally split across GPUs

In [ ]:
#@title 🎧 Code Walkthrough: Col Parallel Split
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_col_parallel_split.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Column-linear parallelism with T=2 GPUs
T = 2  # tensor parallelism degree

# Step 1: Split W along columns
assert n % T == 0, f"Output dim {n} must be divisible by T={T}"
cols_per_gpu = n // T

W_shards = torch.chunk(W, T, dim=1)  # Split along columns (dim=1)

print(f"Original W shape: {W.shape}")
for i, shard in enumerate(W_shards):
    print(f"GPU {i} weight shard shape: {shard.shape}")
print(f"\nEach GPU stores {cols_per_gpu} columns out of {n} total")
print(f"Memory per GPU: {100/T:.0f}% of the full weight matrix")

In [ ]:
#@title 🎧 Code Walkthrough: Col Parallel Compute
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_col_parallel_compute.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 2: Each GPU computes its slice of the output independently
# X is replicated -- every GPU has the full input

partial_outputs = []
for i in range(T):
    Y_i = X @ W_shards[i]  # GPU i computes X @ W_i
    partial_outputs.append(Y_i)
    print(f"GPU {i}: Y_{i} shape = {Y_i.shape}")

# Step 3: Concatenate partial outputs along the last dimension
Y_column_parallel = torch.cat(partial_outputs, dim=1)
print(f"\nConcatenated output shape: {Y_column_parallel.shape}")

# Verify correctness
print(f"\nMatches full computation: {torch.allclose(Y_column_parallel, Y_full, atol=1e-5)}")
print(f"Max difference: {(Y_column_parallel - Y_full).abs().max().item():.2e}")

In [ ]:
#@title 🎧 Listen: Col Parallel Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_col_parallel_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


The outputs match exactly. Notice that **no communication** was needed between the simulated GPUs -- each one computed its part independently, and we only gathered the results at the end.

### Building a Reusable ColumnParallelLinear Class

Let us encapsulate this into a proper PyTorch module.

In [ ]:
#@title 🎧 Code Walkthrough: Col Parallel Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_col_parallel_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class ColumnParallelLinear(nn.Module):
    """Linear layer with weight split along the output dimension (columns).
    
    Each simulated GPU stores only its shard of the weight matrix.
    The input X is replicated, and each GPU computes a slice of the output.
    """
    
    def __init__(self, in_features: int, out_features: int, world_size: int, rank: int):
        super().__init__()
        assert out_features % world_size == 0, \
            f"out_features ({out_features}) must be divisible by world_size ({world_size})"
        
        self.in_features = in_features
        self.out_features = out_features
        self.world_size = world_size
        self.rank = rank
        self.out_features_per_gpu = out_features // world_size
        
        # Each GPU stores only its column shard
        self.weight = nn.Parameter(
            torch.randn(in_features, self.out_features_per_gpu)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: x @ self.weight
        
        Args:
            x: Input tensor of shape (batch, ..., in_features), replicated on all GPUs
        Returns:
            Output of shape (batch, ..., out_features_per_gpu) -- split across GPUs
        """
        return x @ self.weight
    
    @staticmethod
    def from_full_weight(weight: torch.Tensor, world_size: int, rank: int):
        """Create a ColumnParallelLinear from a shard of a full weight matrix."""
        in_features, out_features = weight.shape
        layer = ColumnParallelLinear(in_features, out_features, world_size, rank)
        cols_per_gpu = out_features // world_size
        start_col = rank * cols_per_gpu
        end_col = start_col + cols_per_gpu
        layer.weight = nn.Parameter(weight[:, start_col:end_col].clone())
        return layer

In [ ]:
#@title 🎧 Code Walkthrough: Test Col Parallel Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_test_col_parallel_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Test the ColumnParallelLinear class
T = 2
col_layers = [
    ColumnParallelLinear.from_full_weight(W, world_size=T, rank=i)
    for i in range(T)
]

# Each "GPU" computes its portion
outputs = [layer(X) for layer in col_layers]
Y_col = torch.cat(outputs, dim=-1)

print(f"Column-parallel output shape: {Y_col.shape}")
print(f"Matches full computation: {torch.allclose(Y_col, Y_full, atol=1e-5)}")

In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Column Parallelism with T=4

Extend the column-parallel computation to work with **T=4 simulated GPUs**. Use a larger weight matrix.

In [ ]:
#@title 🎧 Before You Start: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 1: Column parallelism with T=4
# Create a larger weight matrix and split it across 4 GPUs

T_ex1 = 4
k_ex1 = 16
n_ex1 = 32
b_ex1 = 4

torch.manual_seed(123)
X_ex1 = torch.randn(b_ex1, k_ex1, device=device)
W_ex1 = torch.randn(k_ex1, n_ex1, device=device)

# Reference answer
Y_ref = X_ex1 @ W_ex1

# ---- YOUR CODE HERE ----
# 1. Create 4 ColumnParallelLinear layers from the full weight
# 2. Run each layer on X_ex1
# 3. Concatenate the outputs
# 4. Verify the result matches Y_ref



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Listen: Todo1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_todo1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 1
col_layers_ex1 = [
    ColumnParallelLinear.from_full_weight(W_ex1, world_size=T_ex1, rank=i)
    for i in range(T_ex1)
]

outputs_ex1 = [layer(X_ex1) for layer in col_layers_ex1]
Y_col_ex1 = torch.cat(outputs_ex1, dim=-1)

print(f"Weight per GPU: {W_ex1.shape} -> {col_layers_ex1[0].weight.shape}")
print(f"Output per GPU: {outputs_ex1[0].shape}")
print(f"Final output: {Y_col_ex1.shape}")
print(f"Matches reference: {torch.allclose(Y_col_ex1, Y_ref, atol=1e-5)}")

---


In [ ]:
#@title 🎧 Transition: Section3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_section3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 3: Row-Linear Parallelism

### The Idea

Now we split $W$ along its **row dimension**:

$$W = \begin{bmatrix} W_1 \\ W_2 \\ \vdots \\ W_T \end{bmatrix}$$

Each $W_i$ has shape $(k/T, n)$. But here is the catch: if $W$ is split along rows, the **input $X$ must also be split** along its feature dimension.

GPU $i$ gets input slice $X_i$ of shape $(b, k/T)$ and weight shard $W_i$ of shape $(k/T, n)$.

Each GPU computes a **partial product**:
$$Y_i = X_i W_i \quad \text{(shape } (b, n) \text{)}$$

Each $Y_i$ has the full output width, but is only a partial sum. The full output is:
$$Y = \sum_{i=1}^{T} Y_i$$

This summation across GPUs is an **allreduce** operation.

**Key properties:**
- Input must be split across GPUs along its feature dimension
- Allreduce (sum) is required to combine partial sums
- Output is replicated on every GPU after the allreduce

In [ ]:
#@title 🎧 Code Walkthrough: Row Parallel Split
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_row_parallel_split.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Row-linear parallelism with T=2 GPUs
T = 2

# Step 1: Split W along rows
assert k % T == 0, f"Input dim {k} must be divisible by T={T}"
rows_per_gpu = k // T

W_row_shards = torch.chunk(W, T, dim=0)  # Split along rows (dim=0)

print(f"Original W shape: {W.shape}")
for i, shard in enumerate(W_row_shards):
    print(f"GPU {i} weight shard shape: {shard.shape}")

# Step 2: Split X along its feature dimension
X_splits = torch.chunk(X, T, dim=1)  # Split along features (dim=1)

print(f"\nOriginal X shape: {X.shape}")
for i, split in enumerate(X_splits):
    print(f"GPU {i} input slice shape: {split.shape}")

In [ ]:
#@title 🎧 Code Walkthrough: Row Parallel Compute
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_row_parallel_compute.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 3: Each GPU computes its partial product
partial_sums = []
for i in range(T):
    Y_i = X_splits[i] @ W_row_shards[i]  # (b, k/T) @ (k/T, n) = (b, n)
    partial_sums.append(Y_i)
    print(f"GPU {i}: partial Y_{i} shape = {Y_i.shape}")
    print(f"  Values: {Y_i[0, :4].tolist()}")

# Step 4: Allreduce -- sum all partial products
# In real distributed training, this is dist.all_reduce(tensor, op=ReduceOp.SUM)
# Here we simulate it with a simple sum
Y_row_parallel = sum(partial_sums)

print(f"\nAfter allreduce (sum):")
print(f"Output shape: {Y_row_parallel.shape}")
print(f"Matches full computation: {torch.allclose(Y_row_parallel, Y_full, atol=1e-5)}")
print(f"Max difference: {(Y_row_parallel - Y_full).abs().max().item():.2e}")

Notice the critical difference from column-parallel:
- Column-parallel: **no communication** during forward, outputs are split
- Row-parallel: **allreduce required** during forward, outputs are replicated


In [ ]:
#@title 🎧 Listen: Row Parallel Math
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_row_parallel_math.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Why Does the Math Work?

Let us verify this with the block matrix multiplication identity:

$$Y = X W = \begin{bmatrix} X_1 & X_2 \end{bmatrix} \begin{bmatrix} W_1 \\ W_2 \end{bmatrix} = X_1 W_1 + X_2 W_2$$

This is just the standard rule that multiplying with a block matrix is the same as summing the products of corresponding blocks.

In [ ]:
#@title 🎧 Code Walkthrough: Row Parallel Math Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_row_parallel_math_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Mathematical verification
# Y = X @ W should equal X_1 @ W_1 + X_2 @ W_2

X_1, X_2 = X[:, :k//2], X[:, k//2:]
W_1, W_2 = W[:k//2, :], W[k//2:, :]

Y_manual = X_1 @ W_1 + X_2 @ W_2
print(f"X @ W == X_1 @ W_1 + X_2 @ W_2: {torch.allclose(Y_full, Y_manual, atol=1e-5)}")
print(f"\nThis is exactly what row-parallel computes!")
print(f"Each GPU computes one term of the sum, then allreduce adds them.")

In [ ]:
#@title 🎧 Code Walkthrough: Row Parallel Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_row_parallel_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Building a Reusable RowParallelLinear Class

In [ ]:
class RowParallelLinear(nn.Module):
    """Linear layer with weight split along the input dimension (rows).
    
    Each simulated GPU stores only its row shard of the weight matrix.
    The input must be split, and an allreduce is needed to combine partial sums.
    """
    
    def __init__(self, in_features: int, out_features: int, world_size: int, rank: int):
        super().__init__()
        assert in_features % world_size == 0, \
            f"in_features ({in_features}) must be divisible by world_size ({world_size})"
        
        self.in_features = in_features
        self.out_features = out_features
        self.world_size = world_size
        self.rank = rank
        self.in_features_per_gpu = in_features // world_size
        
        # Each GPU stores only its row shard
        self.weight = nn.Parameter(
            torch.randn(self.in_features_per_gpu, out_features)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: x @ self.weight (produces a partial sum).
        
        Args:
            x: Input of shape (batch, ..., in_features_per_gpu) -- split across GPUs
        Returns:
            Partial output of shape (batch, ..., out_features) -- needs allreduce!
        """
        return x @ self.weight
    
    @staticmethod
    def from_full_weight(weight: torch.Tensor, world_size: int, rank: int):
        """Create a RowParallelLinear from a shard of a full weight matrix."""
        in_features, out_features = weight.shape
        layer = RowParallelLinear(in_features, out_features, world_size, rank)
        rows_per_gpu = in_features // world_size
        start_row = rank * rows_per_gpu
        end_row = start_row + rows_per_gpu
        layer.weight = nn.Parameter(weight[start_row:end_row, :].clone())
        return layer


def simulated_allreduce(tensors: list) -> torch.Tensor:
    """Simulate allreduce (sum) across GPUs.
    
    In real distributed training, this would be dist.all_reduce(tensor, op=ReduceOp.SUM).
    Here, we just sum the list of tensors.
    """
    return sum(tensors)

In [ ]:
#@title 🎧 Code Walkthrough: Test Row Parallel Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_test_row_parallel_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Test the RowParallelLinear class
T = 2

row_layers = [
    RowParallelLinear.from_full_weight(W, world_size=T, rank=i)
    for i in range(T)
]

# Split input for row-parallel
X_splits = torch.chunk(X, T, dim=-1)

# Each GPU computes its partial result
partial_results = [layer(X_splits[i]) for i, layer in enumerate(row_layers)]

# Allreduce to get the final answer
Y_row = simulated_allreduce(partial_results)

print(f"Row-parallel output shape: {Y_row.shape}")
print(f"Matches full computation: {torch.allclose(Y_row, Y_full, atol=1e-5)}")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Row Parallelism with T=4

Implement row-parallel computation with T=4 GPUs. Remember: both the weight matrix *and* the input need to be split.

In [ ]:
#@title 🎧 Before You Start: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Row parallelism with T=4

T_ex2 = 4
k_ex2 = 16  # input features (must be divisible by T=4)
n_ex2 = 8   # output features
b_ex2 = 3

torch.manual_seed(456)
X_ex2 = torch.randn(b_ex2, k_ex2, device=device)
W_ex2 = torch.randn(k_ex2, n_ex2, device=device)

# Reference
Y_ref2 = X_ex2 @ W_ex2

# ---- YOUR CODE HERE ----
# 1. Create 4 RowParallelLinear layers from the full weight
# 2. Split X_ex2 along the feature dimension into 4 parts
# 3. Run each layer on its corresponding input split
# 4. Allreduce (sum) the partial results
# 5. Verify the result matches Y_ref2



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Listen: Todo2 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_todo2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 2
row_layers_ex2 = [
    RowParallelLinear.from_full_weight(W_ex2, world_size=T_ex2, rank=i)
    for i in range(T_ex2)
]

X_splits_ex2 = torch.chunk(X_ex2, T_ex2, dim=-1)
partial_ex2 = [layer(X_splits_ex2[i]) for i, layer in enumerate(row_layers_ex2)]
Y_row_ex2 = simulated_allreduce(partial_ex2)

print(f"Weight per GPU: {W_ex2.shape} -> {row_layers_ex2[0].weight.shape}")
print(f"Input per GPU: {X_ex2.shape} -> {X_splits_ex2[0].shape}")
print(f"Partial output per GPU: {partial_ex2[0].shape}")
print(f"Final output: {Y_row_ex2.shape}")
print(f"Matches reference: {torch.allclose(Y_row_ex2, Y_ref2, atol=1e-5)}")

---


In [ ]:
#@title 🎧 Transition: Section4 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_section4_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 4: Comparing Column-Parallel vs Row-Parallel

Let us build a clear comparison of the two approaches.

In [ ]:
#@title 🎧 Code Walkthrough: Comparison Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_comparison_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Side-by-side comparison
T = 2
h = 64   # hidden dimension
n_out = 128  # output dimension
batch = 4

torch.manual_seed(789)
X_cmp = torch.randn(batch, h, device=device)
W_cmp = torch.randn(h, n_out, device=device)
Y_ref_cmp = X_cmp @ W_cmp

print("=" * 60)
print("COLUMN-PARALLEL")
print("=" * 60)
print(f"Weight split: ({h}, {n_out}) -> {T} x ({h}, {n_out//T})")
print(f"Input: replicated on all GPUs, shape ({batch}, {h})")
print(f"Output per GPU: ({batch}, {n_out//T})")
print(f"Combine: concatenate along last dim")
print(f"Communication: NONE")

print()
print("=" * 60)
print("ROW-PARALLEL")
print("=" * 60)
print(f"Weight split: ({h}, {n_out}) -> {T} x ({h//T}, {n_out})")
print(f"Input: split across GPUs, ({batch}, {h}) -> {T} x ({batch}, {h//T})")
print(f"Output per GPU: ({batch}, {n_out}) -- full width, but partial sum!")
print(f"Combine: allreduce (sum)")
print(f"Communication: ONE ALLREDUCE")

# Verify both give the same answer
col_layers_cmp = [ColumnParallelLinear.from_full_weight(W_cmp, T, i) for i in range(T)]
Y_col_cmp = torch.cat([l(X_cmp) for l in col_layers_cmp], dim=-1)

row_layers_cmp = [RowParallelLinear.from_full_weight(W_cmp, T, i) for i in range(T)]
X_splits_cmp = torch.chunk(X_cmp, T, dim=-1)
Y_row_cmp = simulated_allreduce([l(X_splits_cmp[i]) for i, l in enumerate(row_layers_cmp)])

print(f"\nColumn-parallel matches: {torch.allclose(Y_col_cmp, Y_ref_cmp, atol=1e-5)}")
print(f"Row-parallel matches:    {torch.allclose(Y_row_cmp, Y_ref_cmp, atol=1e-5)}")

---


In [ ]:
#@title 🎧 Transition: Section5 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_section5_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Section 5: Why This Matters -- Memory Savings

Let us calculate the memory savings for a realistic model.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_28_memory_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Memory analysis for a realistic transformer
h = 12288       # GPT-3 175B hidden dimension
intermediate = 4 * h  # MLP intermediate dimension

# Weight sizes (in parameters)
w1_params = h * intermediate  # MLP first layer
w2_params = intermediate * h  # MLP second layer
wq_params = h * h             # Q projection
wk_params = h * h             # K projection
wv_params = h * h             # V projection
wo_params = h * h             # Output projection

total_params_per_layer = w1_params + w2_params + wq_params + wk_params + wv_params + wo_params

print(f"Hidden dimension h = {h}")
print(f"Intermediate dimension 4h = {intermediate}")
print(f"\nParameters per transformer layer:")
print(f"  MLP W1 (h x 4h):  {w1_params:>12,} = {w1_params * 2 / 1e9:.2f} GB (FP16)")
print(f"  MLP W2 (4h x h):  {w2_params:>12,} = {w2_params * 2 / 1e9:.2f} GB (FP16)")
print(f"  Attention Q,K,V,O: {4 * h * h:>12,} = {4 * h * h * 2 / 1e9:.2f} GB (FP16)")
print(f"  Total per layer:   {total_params_per_layer:>12,} = {total_params_per_layer * 2 / 1e9:.2f} GB (FP16)")

print(f"\n{'T (GPUs)':>10} {'Params/GPU':>15} {'Memory/GPU (FP16)':>20}")
print("-" * 50)
for T in [1, 2, 4, 8]:
    params_per_gpu = total_params_per_layer // T
    mem_per_gpu = params_per_gpu * 2 / 1e9
    print(f"{T:>10} {params_per_gpu:>15,} {mem_per_gpu:>17.2f} GB")

In [ ]:
#@title 🎧 Listen: Memory Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_29_memory_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


With T=8 GPUs, each GPU stores only 1/8 of the weight matrices. This is crucial for models like GPT-3 (96 layers, ~175B parameters), where the weights alone would not fit on a single GPU.


In [ ]:
#@title 🎧 Transition: Section6 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_30_section6_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
## Section 6: The Chain -- Column Followed by Row

The real power of tensor parallelism comes from chaining column-parallel and row-parallel layers together. This is exactly what Megatron-LM does in the MLP block.

The key insight: the output of a column-parallel layer is **split** across GPUs -- which is exactly the input format a row-parallel layer needs!

In [ ]:
#@title 🎧 Code Walkthrough: Chain Demo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_31_chain_demo.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Demonstrate the column -> row chain
T = 2
h = 16
intermediate = 4 * h  # = 64
batch = 2

torch.manual_seed(42)
X_chain = torch.randn(batch, h, device=device)
W1_full = torch.randn(h, intermediate, device=device)   # First layer
W2_full = torch.randn(intermediate, h, device=device)    # Second layer

# Reference: full computation
Z_full = X_chain @ W1_full                  # (batch, intermediate)
Z_full_gelu = torch.nn.functional.gelu(Z_full)
Y_full_chain = Z_full_gelu @ W2_full        # (batch, h)

print("=== Full (non-parallel) computation ===")
print(f"X -> W1 -> GeLU -> W2 -> Y")
print(f"X: {X_chain.shape} -> Z: {Z_full.shape} -> Y: {Y_full_chain.shape}")

print("\n=== Tensor-parallel computation (T=2) ===")

# Step 1: Column-parallel first layer
col_layers_chain = [ColumnParallelLinear.from_full_weight(W1_full, T, i) for i in range(T)]

# Each GPU computes its slice of the intermediate activations
Z_parts = [layer(X_chain) for layer in col_layers_chain]
for i, z in enumerate(Z_parts):
    print(f"GPU {i} after column-parallel W1: {z.shape}")

# Step 2: GeLU activation -- LOCAL, no communication!
Z_gelu_parts = [torch.nn.functional.gelu(z) for z in Z_parts]
print(f"\nGeLU applied locally on each GPU (element-wise, no communication)")

# Step 3: Row-parallel second layer
# The split activations from column-parallel are exactly what row-parallel needs!
row_layers_chain = [RowParallelLinear.from_full_weight(W2_full, T, i) for i in range(T)]
Y_parts = [layer(Z_gelu_parts[i]) for i, layer in enumerate(row_layers_chain)]
for i, y in enumerate(Y_parts):
    print(f"GPU {i} after row-parallel W2 (partial sum): {y.shape}")

# Step 4: Allreduce
Y_chain = simulated_allreduce(Y_parts)
print(f"\nAfter allreduce: {Y_chain.shape}")
print(f"Matches full computation: {torch.allclose(Y_chain, Y_full_chain, atol=1e-4)}")
print(f"Max difference: {(Y_chain - Y_full_chain).abs().max().item():.2e}")

In [ ]:
#@title 🎧 Listen: Chain Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_32_chain_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


Notice what happened:
1. Column-parallel W1 produced **split** activations (each GPU has shape `(batch, intermediate/T)`)
2. GeLU is element-wise, so it works on split data with **no communication**
3. Row-parallel W2 *needs* split input -- and that is exactly what it gets!
4. The only communication is **one allreduce** at the very end

This is the Megatron-LM insight: column-parallel first, row-parallel second, with only a single allreduce for the entire MLP block.


In [ ]:
#@title 🎧 Transition: Section7 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_33_section7_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
## Section 7: Communication Cost Analysis

In [ ]:
#@title 🎧 Code Walkthrough: Comm Cost Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_34_comm_cost_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Communication cost analysis

def analyze_allreduce_cost(batch_size, seq_len, hidden_dim, num_layers, tp_degree, dtype_bytes=2):
    """Analyze the communication cost of tensor parallelism."""
    # Each allreduce transfers a tensor of shape (batch, seq_len, hidden_dim)
    tensor_size_bytes = batch_size * seq_len * hidden_dim * dtype_bytes
    tensor_size_mb = tensor_size_bytes / (1024**2)
    
    # Two allreduces per layer (one for attention, one for MLP)
    allreduces_per_pass = 2 * num_layers
    total_data_per_pass = allreduces_per_pass * tensor_size_bytes
    total_data_gb = total_data_per_pass / (1024**3)
    
    # Ring allreduce transfers approximately 2x the data
    ring_factor = 2 * (tp_degree - 1) / tp_degree
    actual_data_gb = total_data_gb * ring_factor
    
    return {
        'tensor_size_mb': tensor_size_mb,
        'allreduces_per_forward': allreduces_per_pass,
        'total_data_forward_gb': total_data_gb,
        'ring_adjusted_gb': actual_data_gb,
    }

# GPT-3 175B parameters
configs = [
    ('GPT-3 175B', 1, 2048, 12288, 96, 8),
    ('LLaMA 70B', 1, 4096, 8192, 80, 8),
    ('Small model', 4, 1024, 2048, 24, 4),
]

for name, b, s, h, L, T in configs:
    result = analyze_allreduce_cost(b, s, h, L, T)
    print(f"\n{'='*50}")
    print(f"{name} (T={T})")
    print(f"{'='*50}")
    print(f"  Tensor per allreduce: {result['tensor_size_mb']:.1f} MB")
    print(f"  Allreduces per forward: {result['allreduces_per_forward']}")
    print(f"  Total data per forward: {result['total_data_forward_gb']:.2f} GB")
    print(f"  Ring-adjusted volume:   {result['ring_adjusted_gb']:.2f} GB")
    
    # NVLink vs InfiniBand comparison
    nvlink_bw = 300  # GB/s per GPU
    ib_bw = 25       # GB/s
    nvlink_time = result['ring_adjusted_gb'] / nvlink_bw * 1000  # ms
    ib_time = result['ring_adjusted_gb'] / ib_bw * 1000          # ms
    print(f"  Time (NVLink 300 GB/s): {nvlink_time:.1f} ms")
    print(f"  Time (IB 25 GB/s):      {ib_time:.1f} ms")
    print(f"  Speedup of NVLink:      {ib_time/nvlink_time:.1f}x")

In [ ]:
#@title 🎧 Listen: Comm Cost Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_35_comm_cost_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


This is why tensor parallelism is a **within-node** strategy. On NVLink, the allreduce overhead is manageable. Over network interconnects, it would dominate training time.


In [ ]:
#@title 🎧 Transition: Section8 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_36_section8_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
## Section 8: Backward Pass -- Gradients in Tensor Parallelism

The forward pass is only half the story. Let us verify that gradients flow correctly through our parallel layers.

In [ ]:
#@title 🎧 Code Walkthrough: Col Grad Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_37_col_grad_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Gradient verification for column-parallel
T = 2
h_grad = 8
n_grad = 12
b_grad = 2

torch.manual_seed(42)

# Full computation with gradients
W_full_grad = torch.randn(h_grad, n_grad, device=device, requires_grad=True)
X_grad = torch.randn(b_grad, h_grad, device=device)
Y_full_grad = X_grad @ W_full_grad
loss_full = Y_full_grad.sum()
loss_full.backward()
full_grad = W_full_grad.grad.clone()

# Column-parallel computation
W_copy = W_full_grad.data.clone()
col_layers_grad = []
for i in range(T):
    layer = ColumnParallelLinear.from_full_weight(W_copy, T, i)
    layer.weight.requires_grad_(True)
    col_layers_grad.append(layer)

outputs_grad = [layer(X_grad) for layer in col_layers_grad]
Y_col_grad = torch.cat(outputs_grad, dim=-1)
loss_col = Y_col_grad.sum()
loss_col.backward()

# Reconstruct full gradient from shards
reconstructed_grad = torch.cat([l.weight.grad for l in col_layers_grad], dim=1)

print(f"Full gradient shape: {full_grad.shape}")
print(f"Reconstructed gradient shape: {reconstructed_grad.shape}")
print(f"Gradients match: {torch.allclose(full_grad, reconstructed_grad, atol=1e-5)}")
print(f"\nEach GPU computes the gradient for its shard independently!")
for i, layer in enumerate(col_layers_grad):
    print(f"GPU {i} gradient shape: {layer.weight.grad.shape}")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_38_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 3: Gradient Verification for Row-Parallel

Verify that gradients flow correctly through a row-parallel layer. Remember that the forward pass requires split inputs and an allreduce.

In [ ]:
#@title 🎧 Before You Start: Todo3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_39_todo3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 3: Verify gradients for row-parallel

T_ex3 = 2
h_ex3 = 8
n_ex3 = 6
b_ex3 = 2

torch.manual_seed(99)

# Full computation (reference)
W_ref_grad = torch.randn(h_ex3, n_ex3, device=device, requires_grad=True)
X_ref_grad = torch.randn(b_ex3, h_ex3, device=device)
Y_ref_grad = X_ref_grad @ W_ref_grad
loss_ref = Y_ref_grad.sum()
loss_ref.backward()
ref_grad = W_ref_grad.grad.clone()

# ---- YOUR CODE HERE ----
# 1. Create T=2 RowParallelLinear layers from W_ref_grad.data
# 2. Split X_ref_grad along feature dim
# 3. Forward pass through each layer
# 4. Sum partial results (simulated allreduce)
# 5. Compute loss and backward
# 6. Reconstruct the full gradient by stacking row shards
# 7. Compare with ref_grad



# ---- END YOUR CODE ----

In [ ]:
#@title 🎧 Listen: Todo3 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_40_todo3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Solution for Exercise 3
W_data = W_ref_grad.data.clone()

row_layers_grad = []
for i in range(T_ex3):
    layer = RowParallelLinear.from_full_weight(W_data, T_ex3, i)
    layer.weight.requires_grad_(True)
    row_layers_grad.append(layer)

X_splits_grad = torch.chunk(X_ref_grad, T_ex3, dim=-1)
partial_grads = [layer(X_splits_grad[i]) for i, layer in enumerate(row_layers_grad)]
Y_row_grad = sum(partial_grads)  # Simulated allreduce
loss_row = Y_row_grad.sum()
loss_row.backward()

# Reconstruct gradient by stacking row shards
reconstructed_row_grad = torch.cat([l.weight.grad for l in row_layers_grad], dim=0)

print(f"Reference gradient shape: {ref_grad.shape}")
print(f"Reconstructed gradient shape: {reconstructed_row_grad.shape}")
print(f"Gradients match: {torch.allclose(ref_grad, reconstructed_row_grad, atol=1e-5)}")
for i, layer in enumerate(row_layers_grad):
    print(f"GPU {i} gradient shape: {layer.weight.grad.shape}")

In [ ]:
#@title 🎧 Wrap-Up: Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_41_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


---
## Summary

In this notebook, you implemented and verified the two fundamental building blocks of tensor parallelism:

| Property | Column-Parallel | Row-Parallel |
|----------|----------------|---------------|
| Weight split | Along columns (output dim) | Along rows (input dim) |
| Input requirement | Replicated | Split |
| Output format | Split across GPUs | Partial sums (full width) |
| Communication | None | Allreduce (sum) |
| Combine operation | Concatenate | Sum |

**Key takeaway:** When you chain column-parallel (first layer) and row-parallel (second layer), the intermediate split format matches naturally, and you need only **one allreduce** for the entire pair. This is the Megatron-LM insight that makes tensor parallelism practical.

In the next notebook, we will use these building blocks to implement a complete Megatron-style MLP block.